<a href="https://colab.research.google.com/github/monishabhojkumar12-avi/Aerial-Object-classification-and-detection/blob/main/Aerial_object.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/classification_dataset.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully ✅")

Unzipped successfully ✅


In [ ]:
import os
os.listdir("/content/classification_dataset")

['valid', 'train', 'test']

In [ ]:
os.listdir("/content/classification_dataset/train")

['drone', 'bird']

In [ ]:
def count_images(path):
    for cls in os.listdir(path):
        class_path = os.path.join(path, cls)
        if os.path.isdir(class_path):
            print(f"{cls}: {len(os.listdir(class_path))}")

print("Train Data:")
count_images("/content/classification_dataset/train")

print("\nValidation Data:")
count_images("/content/classification_dataset/valid")

print("\nTest Data:")
count_images("/content/classification_dataset/test")

Train Data:
drone: 1248
bird: 1414

Validation Data:
drone: 225
bird: 217

Test Data:
drone: 94
bird: 121


In [ ]:
import os

print("Train:", os.listdir("/content/classification_dataset/train"))
print("Valid:", os.listdir("/content/classification_dataset/valid"))
print("Test:", os.listdir("/content/classification_dataset/test"))

Train: ['drone', 'bird']
Valid: ['drone', 'bird']
Test: ['drone', 'bird']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_data = image_dataset_from_directory(
    "/content/classification_dataset/train",
    image_size=(224,224),
    batch_size=32
)

val_data = image_dataset_from_directory(
    "/content/classification_dataset/valid",
    image_size=(224,224),
    batch_size=32
)

Found 2662 files belonging to 2 classes.
Found 442 files belonging to 2 classes.


In [ ]:
from tensorflow.keras import models, layers

model = models.Sequential([

    layers.Input(shape=(224,224,3)),

    # Normalize
    layers.Rescaling(1./255),

    # -------- BLOCK 1 --------
    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- BLOCK 2 --------
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- BLOCK 3 --------
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- BLOCK 4 (NEW - important) --------
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # -------- HEAD --------
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    # Output
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_3 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             

 Total params: 1,275,937 (4.87 MB)

 Trainable params: 1,273,505 (4.86 MB)

 Non-trainable params: 2,432 (9.50 KB)

In [ ]:
history = model.fit(train_data, validation_data=val_data, epochs=15)

Epoch 1/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 881s 10s/step - accuracy: 0.8576 - loss: 0.3479 - val_accuracy: 0.6606 - val_loss: 0.7020
Epoch 2/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 873s 10s/step - accuracy: 0.8516 - loss: 0.3299 - val_accuracy: 0.6176 - val_loss: 0.9036
Epoch 3/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 881s 10s/step - accuracy: 0.8565 - loss: 0.3343 - val_accuracy: 0.7760 - val_loss: 0.5205
Epoch 4/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 936s 11s/step - accuracy: 0.8805 - loss: 0.2848 - val_accuracy: 0.7941 - val_loss: 0.4468
Epoch 5/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1003s 11s/step - accuracy: 0.8764 - loss: 0.3003 - val_accuracy: 0.8145 - val_loss: 0.4148
Epoch 6/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 889s 11s/step - accuracy: 0.8911 - loss: 0.2618 - val_accuracy: 0.7036 - val_loss: 0.9920
Epoch 7/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 896s 11s/step - accuracy: 0.8877 - loss: 0.2639 - val_accuracy: 0.6719 - val_loss: 0.7408
Epoch 8/15
84/84 ━━━━━━━━━━━━━━━━━━━━ 1029s 12s/step - accuracy: 0.8941 - loss: 0.2625 - val_accuracy: 

In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/cnn(1)_model.h5")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


In [ ]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
train_data = train_gen.flow_from_directory(
    "/content/classification_dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    "/content/classification_dataset/valid",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

Found 2662 images belonging to 2 classes.
Found 442 images belonging to 2 classes.


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)


base_model.trainable = False


x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

cnn_model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,591,297 (9.89 MB)

 Trainable params: 1,536,833 (5.86 MB)

 Non-trainable params: 1,054,464 (4.02 MB)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# -------------------------
# 1. Load pretrained MobileNet
# -------------------------
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# -------------------------
# 2. Build model
# -------------------------
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# -------------------------
# 3. Compile (initial training)
# -------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -------------------------
# 4. Callbacks
# -------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor='val_loss',
    save_best_only=True
)

# -------------------------
# 5. Train (initial)
# -------------------------
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

# -------------------------
# 6. Fine-tuning
# -------------------------
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Recompile with low learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -------------------------
# 7. Train again (fine-tune)
# -------------------------
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    callbacks=[early_stop, checkpoint]
)

# -------------------------
# 8. Save final model
# -------------------------
model.save("mobilenet(1)_model.h5")

Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8984 - loss: 0.2259

84/84 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.9395 - loss: 0.1589 - val_accuracy: 0.9751 - val_loss: 0.0996
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9788 - loss: 0.0542

84/84 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.9741 - loss: 0.0648 - val_accuracy: 0.9774 - val_loss: 0.0763
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 180s 2s/step - accuracy: 0.9895 - loss: 0.0327 - val_accuracy: 0.9706 - val_loss: 0.0879
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9841 - loss: 0.0361

84/84 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.9861 - loss: 0.0358 - val_accuracy: 0.9819 - val_loss: 0.0756
Epoch 5/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 179s 2s/step - accuracy: 0.9917 - loss: 0.0218 - val_accuracy: 0.9751 - val_loss: 0.1052
Epoch 6/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 179s 2s/step - accuracy: 0.9887 - loss: 0.0273 - val_accuracy: 0.9796 - val_loss: 0.0909
Epoch 7/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 179s 2s/step - accuracy: 0.9906 - loss: 0.0242 - val_accuracy: 0.9819 - val_loss: 0.0885
Epoch 1/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.9688 - loss: 0.0913 - val_accuracy: 0.9796 - val_loss: 0.0781
Epoch 2/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 210s 3s/step - accuracy: 0.9838 - loss: 0.0441 - val_accuracy: 0.9751 - val_loss: 0.0841
Epoch 3/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 209s 2s/step - accuracy: 0.9872 - loss: 0.0327 - val_accuracy: 0.9751 - val_loss: 0.0869


In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/mobilenet(2)_model.h5")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint("best_mobilenet.h5", save_best_only=True)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.8287 - loss: 0.3695 

84/84 ━━━━━━━━━━━━━━━━━━━━ 1043s 12s/step - accuracy: 0.8242 - loss: 0.3841 - val_accuracy: 0.4910 - val_loss: 0.7330
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 997s 12s/step - accuracy: 0.8509 - loss: 0.3482 - val_accuracy: 0.4910 - val_loss: 1.9513
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 966s 11s/step - accuracy: 0.8565 - loss: 0.3318 - val_accuracy: 0.3552 - val_loss: 0.7609
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 957s 11s/step - accuracy: 0.8535 - loss: 0.3336 - val_accuracy: 0.4977 - val_loss: 1.0562


In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/mobilenet(1)_model.h5")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

IMG_SIZE = 224
BATCH_SIZE = 32

# Load dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "classification_dataset/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "classification_dataset/valid", # Changed 'val' to 'valid'
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

num_classes = len(train_ds.class_names)

# Load ResNet50 base model
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # Freeze base model

# Build model
model = models.Sequential([
    layers.Rescaling(1./255),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save
#model.save("resnet_model.h5")

Found 2662 files belonging to 2 classes.
Found 442 files belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 553s 6s/step - accuracy: 0.5612 - loss: 0.7128 - val_accuracy: 0.6652 - val_loss: 0.6179
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 543s 6s/step - accuracy: 0.6352 - loss: 0.6300 - val_accuracy: 0.6855 - val_loss: 0.5965
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 568s 6s/step - accuracy: 0.6427 - loss: 0.6181 - val_accuracy: 0.6923 - val_loss: 0.5823
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 558s 6s/step - accuracy: 0.6698 - loss: 0.6001 - val_accuracy: 0.6968 - val_loss: 0.5718
Epoch 5/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 519s 6s/step - accuracy: 0.6672 - loss: 0.5914 - val_accuracy: 0.7014 - val_loss: 0.5616
Epoch 6/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 562s 6s/step - accuracy: 0.6769 - loss: 0.5744 - val_accuracy: 0.6946 - val_loss: 0.5591
Epoch 7/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 519s 6s/step - accuracy: 0.6792 - loss: 0.5765 - val_accuracy: 0.705

In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/resnet_model.h5")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = 224
BATCH_SIZE = 32

# Load dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/classification_dataset/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/classification_dataset/valid",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

num_classes = len(train_ds.class_names)

# Load EfficientNet
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # Freeze base

# Build model
model = models.Sequential([
    layers.Rescaling(1./255),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save
model.save("efficientnet_model.h5")

Found 2662 files belonging to 2 classes.
Found 442 files belonging to 2 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 260s 3s/step - accuracy: 0.5267 - loss: 0.9109 - val_accuracy: 0.5090 - val_loss: 0.6942
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 251s 3s/step - accuracy: 0.5229 - loss: 0.8274 - val_accuracy: 0.5090 - val_loss: 0.6917
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 248s 3s/step - accuracy: 0.5533 - loss: 0.7391 - val_accuracy: 0.5566 - val_loss: 0.6914
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 250s 3s/step - accuracy: 0.5398 - loss: 0.7272 - val_accuracy: 0.5113 - val_loss: 0.6916
Epoch 5/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 231s 3s/step - accuracy: 0.5571 - loss: 0.7051 - val_accuracy: 0.5724 - val_loss: 0.6922
Epoch 6/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 239s 3s/step - accuracy: 0.5687 - loss: 0.6926 - val_accuracy: 0.6765 - val_loss: 0.6900
Epoch 7/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 239s 3s/step - accuracy: 0.5702 - loss: 0.6892 - val_accuracy: 0.547

In [ ]:
model.save("/content/drive/MyDrive/aerial_dataset/efficientnet_model.h5")

In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-04-17 12:13:21--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-17 12:13:21--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-17T12%3A56%3A45Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-17T1

In [ ]:
%%writefile model.py
import streamlit as st
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

@st.cache_resource
def load_models():
    cnn = load_model("/content/drive/MyDrive/aerial_dataset/cnn_model.h5")
    mobilenet = load_model("/content/drive/MyDrive/aerial_dataset/mobilenet(2)_model.h5")
    return cnn, mobilenet

cnn_model, mobilenet_model = load_models()


st.set_page_config(page_title="Aerial Classifier", layout="centered")

st.title("🛰️ Aerial Object Classifier")
st.write("Upload an image to classify Bird 🐦 or Drone 🛸")


model_choice = st.selectbox(
    "Choose Model",
    ["CNN Model", "MobileNet Model"]
)

uploaded_file = st.file_uploader("Upload Image", type=["jpg", "png", "jpeg"])


if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded Image", use_column_width=True)


    if st.button("🔍 Predict"):

        img = image.resize((224,224))
        img = np.array(img)


        if model_choice == "CNN Model":
            img = img / 255.0
            model = cnn_model
        else:
            img = preprocess_input(img)
            model = mobilenet_model

        img = np.expand_dims(img, axis=0)


        with st.spinner("Predicting..."):
            prediction = model.predict(img)[0][0]


        st.subheader("📌 Prediction Result")

        if prediction > 0.5:
            label = "🛸 Drone"
            confidence = prediction
        else:
            label = "🐦 Bird"
            confidence = 1 - prediction

        st.success(f"Prediction: {label}")
        st.write(f"Confidence: {confidence:.2f}")

Overwriting model.py


In [ ]:
!streamlit run /content/model.py &>/content/logs.txt &

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://mba-christine-specialized-geometry.trycloudflare.com
